In [2]:
%pip install -q google-genai sentence-transformers tqdm pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import gc
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Data Cleaning
df = pd.read_csv("all_kdrama_threads.csv", engine="python", on_bad_lines="skip")
df = df.dropna(subset=["text"])
df = df[~df["text"].isin(["[deleted]", "[removed]"])]
df = df[df["text"].str.len() >= 80].copy()

# Selecting top 2500 most liked comments per drama (12500 total for 5 dramas)
candidates = (
    df.sort_values("score", ascending=False)
    .groupby("drama")
    .head(2500)
    .reset_index(drop=True)
)

del df
gc.collect()

# Semantic Embedding Search
embedder = SentenceTransformer("all-MiniLM-L6-v2")

query_concept = (
    "Korean beauty standards, physical appearance, lookism, plastic surgery, "
    "weight loss, makeup transformation, body image insecurity, and how society "
    "judges people based on attractiveness."
)
query_emb = embedder.encode(query_concept, convert_to_tensor=True)

text_embs = embedder.encode(
    candidates["text"].tolist(),
    batch_size=128,
    convert_to_tensor=True,
)

candidates["semantic_score"] = (
    util.cos_sim(query_emb, text_embs)[0].cpu().numpy()
)

# Filtering the comments which have a 0.35+ threshold
threshold = 0.35
relevant_df = (
    candidates[candidates["semantic_score"] >= threshold]
    .sort_values("semantic_score", ascending=False)
    .reset_index(drop=True)
)

year_map = {
    "Boys Over Flowers": 2009,
    "Birth Of A Beauty": 2014,
    "My Id Is Gangnam Beauty": 2018,
    "True Beauty": 2020,
    "Mask Girl": 2023,
}
relevant_df["year"] = relevant_df["drama"].map(year_map)
relevant_df.to_csv("all_relevant_beauty_discussions.csv", index=False)

print(
    f"Total relevant comments extracted (score >= {threshold}): {len(relevant_df):,}\n"
)
summary = (
    relevant_df.groupby(["year", "drama"])
    .agg(
        relevant_count=("text", "count"),
        mean_similarity=("semantic_score", "mean"),
        median_upvotes=("score", "median"),
    )
    .round(3)
)
print(summary)

Total relevant comments extracted (score >= 0.35): 613

                              relevant_count  mean_similarity  median_upvotes
year drama                                                                   
2009 Boys Over Flowers                   141            0.414           190.0
2014 Birth Of A Beauty                    95            0.429            27.0
2018 My Id Is Gangnam Beauty             140            0.419           103.0
2020 True Beauty                         155            0.404           233.0
2023 Mask Girl                            82            0.400            38.0
